<a href="https://colab.research.google.com/github/Wolfang-Perez/curso-programacion-cientifica/blob/main/ejercicios/taller_evaluativo_03_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Evaluativo 03 — NumPy (Calidad del Aire PM2.5)

**Estudiante:** Wolfang Felipe Pérez Roldán  
**Fecha:** 23 de octubre de 2026

## Ejercicio 1 — Crear los arreglos y leer su etiqueta

Los datos de la semana se convierten en un arreglo 2D con `np.array` a partir de una
lista de listas: 5 sensores (filas) por 7 días (columnas). Antes de operar con él se
mira su etiqueta, como quien revisa cuántas filas y columnas tiene una hoja de cálculo:
`shape`, `ndim`, `size` y `dtype`.

`dias` se crea con `np.arange(1, 8)`, que incluye el 7, y la tabla de calibración se
guarda en dos arreglos 1D, `pendientes` e `interceptos`, en el orden SBC-A a SBC-E.

In [ ]:
import numpy as np

lecturas_crudas = np.array([
    [10.0, 14.5, 22.0, 30.5, 28.0, 18.0, 9.5],    # SBC-A
    [12.0, 20.0, 35.0, 48.0, 41.0, 25.0, 14.0],   # SBC-B
    [8.0, 11.0, 19.0, 27.0, 33.0, 21.0, 6.0],     # SBC-C
    [15.0, 24.0, 31.0, 44.0, 52.0, 36.0, 17.0],   # SBC-D
    [5.0, 9.0, 16.0, 22.0, 45.0, 30.0, 4.0],      # SBC-E
])

forma = lecturas_crudas.shape
dimensiones = lecturas_crudas.ndim
total = lecturas_crudas.size
tipo = lecturas_crudas.dtype

dias = np.arange(1, 8)

pendientes = np.array([1.2, 0.9, 1.1, 1.0, 1.3])
interceptos = np.array([-0.5, 1.0, -2.0, 0.5, -1.5])

print("Forma:", forma)
print("Dimensiones:", dimensiones)
print("Total de lecturas:", total)
print("Tipo de dato:", tipo)
print("Días:", dias)
print("Pendientes:", pendientes)
print("Interceptos:", interceptos)

## Ejercicio 2 — Señalar celdas y recortar bloques

Con indexación y slicing se leen partes de la tabla, contando desde 0: SBC-A es la
fila 0 y el día 1 es la columna 0.

Se saca una celda (SBC-C en el día 4), una celda con índice negativo (último sensor,
día 1), una columna completa (día 5), una fila completa (la semana de SBC-B) y un
bloque de los sensores SBC-B a SBC-D en los días 3 a 6, que debe quedar de forma
(3, 4). Cada resultado sale de una sola expresión de indexación sobre
`lecturas_crudas`.

In [ ]:
lectura_c_dia4 = lecturas_crudas[2, 3]
ultimo_primer = lecturas_crudas[-1, 0]
dia5 = lecturas_crudas[:, 4]
semana_b = lecturas_crudas[1]
bloque = lecturas_crudas[1:4, 2:6]

print("SBC-C en el día 4:", lectura_c_dia4)
print("Último sensor en el día 1:", ultimo_primer)
print("Lecturas del día 5:", dia5)
print("Semana de SBC-B:", semana_b)
print("Bloque SBC-B a SBC-D, días 3 a 6:")
print(bloque)
print("Forma del bloque:", bloque.shape)

## Ejercicio 3 — Limpiar una campaña con máscaras booleanas

Las ocho lecturas de campaña de SBC-A traen dos centinelas -999.0 y un valor de ruido
por encima de 500.0. Una lectura sirve si está entre 0 y 500.0 inclusive, así que esa
única condición descarta las dos cosas.

`faltantes` marca el centinela y `es_util` combina dos comparaciones entre paréntesis
con `&`. La máscara se usa directamente como índice para quedarse con las 5 lecturas
útiles, y su promedio se calcula con `.mean()`.

In [ ]:
campana_sbc_a = np.array([10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0])

faltantes = campana_sbc_a == -999.0
es_util = (campana_sbc_a >= 0.0) & (campana_sbc_a <= 500.0)

lecturas_utiles = campana_sbc_a[es_util]
promedio_utiles = lecturas_utiles.mean()

print("Lecturas faltantes:", faltantes.sum())
print("Lecturas útiles:", lecturas_utiles)
print("Promedio de las lecturas útiles:", promedio_utiles)

## Ejercicio 4 — Calibrar con operaciones vectorizadas y broadcasting

Cada sensor tiene su propia recta, así que la calibración de las 35 lecturas se hace de
un solo golpe, sin recorrerlas.

**Por qué `lecturas_crudas * pendientes` lanza un `ValueError`:** `lecturas_crudas`
tiene forma (5, 7) y `pendientes` tiene forma (5,). NumPy alinea las formas por la
derecha, de modo que compara el 7 con el 5 en la última dimensión y, como no coinciden
(ni ninguna vale 1), responde `ValueError: operands could not be broadcast together`.
Guardando la pendiente como columna de forma (5, 1) se alinea con las filas: 5 filas
contra 5 pendientes, y las 7 columnas se repiten. Eso es el broadcasting.

La campaña del Ejercicio 3 se calibra con la recta de SBC-A (`pendientes[0]` e
`interceptos[0]`) y su promedio debe dar 31.9, como en el Taller 02.

In [ ]:
pendientes_col = pendientes.reshape(5, 1)
interceptos_col = interceptos[:, np.newaxis]

calibradas = lecturas_crudas * pendientes_col + interceptos_col

campana_calibrada = lecturas_utiles * pendientes[0] + interceptos[0]
promedio_calibrado_sbc_a = campana_calibrada.mean()

print("Forma de calibradas:", calibradas.shape)
print("SBC-A calibrada:", calibradas[0])
print("Campaña calibrada:", campana_calibrada)
print("Promedio calibrado de la campaña SBC-A:", promedio_calibrado_sbc_a)

## Ejercicio 5 — Resumir la semana con estadística por eje

Con `calibradas` se resume la red completa y por dirección. `axis=1` resume por sensor
(un valor por fila) y `axis=0` resume por día (un valor por columna).

Se calculan el promedio general de las 35 lecturas, el promedio y el máximo de cada
sensor, el promedio de cada día y el promedio de las lecturas que superan 35.4
(con una máscara). Todo se muestra redondeado a 2 decimales con `np.round`.

In [ ]:
promedio_general = calibradas.mean()
promedio_por_sensor = calibradas.mean(axis=1)
promedio_por_dia = calibradas.mean(axis=0)
maximo_por_sensor = calibradas.max(axis=1)
media_sobre_limite = calibradas[calibradas > 35.4].mean()

print("Promedio general:", np.round(promedio_general, 2))
print("Promedio por sensor:", np.round(promedio_por_sensor, 2))
print("Promedio por día:", np.round(promedio_por_dia, 2))
print("Máximo por sensor:", np.round(maximo_por_sensor, 2))
print("Media de las lecturas sobre 35.4:", np.round(media_sobre_limite, 2))

## Ejercicio 6 — Funciones universales y reemplazo con máscara

`np.maximum(calibradas - 35.4, 0)` deja 0 donde la lectura está dentro del límite y
cuánto se pasó donde no; sumando cada fila se obtiene el exceso acumulado por sensor.

`np.abs` mide la distancia de cada lectura calibrada de SBC-B al límite de 35.4 y
`en_alerta` es una **copia** de `calibradas` en la que se reemplaza por 0.0 todo lo que
no supera el límite, para que `calibradas` quede intacto. Las celdas de `en_alerta`
mayores que 0 se cuentan con una máscara y `sum`.

In [ ]:
excedente = np.maximum(calibradas - 35.4, 0.0)
excedente_por_sensor = np.round(excedente.sum(axis=1), 1)
desvio_sensor_b = np.round(np.abs(calibradas[1] - 35.4), 1)

en_alerta = calibradas.copy()
en_alerta[en_alerta <= 35.4] = 0.0

print("Excedente por sensor:", excedente_por_sensor)
print("Desvío de SBC-B frente a 35.4:", desvio_sensor_b)
print("Celdas en alerta:", (en_alerta > 0).sum())

## Ejercicio 7 — Integración: el informe de la red

Con máscaras sobre `calibradas` se cuentan las lecturas de cada categoría de la escala
oficial: Buena hasta 12.0, Moderada más de 12.0 y hasta 35.4, Dañina para grupos
sensibles más de 35.4 y hasta 55.4, y Dañina para la salud por encima de 55.4. Cada
intervalo queda abierto por la izquierda, así que ninguna lectura cae en dos categorías
ni se queda sin categoría.

El día más contaminado se obtiene filtrando `dias` con una máscara sobre
`promedio_por_dia` y tomando el primer elemento.

In [ ]:
buenas = (calibradas <= 12.0).sum()
moderadas = ((calibradas > 12.0) & (calibradas <= 35.4)).sum()
daninas_sensibles = ((calibradas > 35.4) & (calibradas <= 55.4)).sum()
daninas_salud = (calibradas > 55.4).sum()

dia_pico = dias[promedio_por_dia == promedio_por_dia.max()][0]

print(f"Lecturas por categoría: Buena={buenas}, Moderada={moderadas}, "
      f"Dañina para grupos sensibles={daninas_sensibles}, "
      f"Dañina para la salud={daninas_salud}")
print(f"Día más contaminado: {dia_pico} con {np.round(promedio_por_dia.max(), 1)} ug/m3")

### Reto de extensión (opcional, sin puntos)

Los mismos cuatro conteos, calculados de otra forma con
`np.digitize(calibradas, [12.0, 35.4, 55.4], right=True)`, que devuelve para cada
lectura un número de 0 a 3 (su categoría). Al final se comprueba que coinciden con los
del Ejercicio 7.

In [ ]:
indices = np.digitize(calibradas, [12.0, 35.4, 55.4], right=True)

reto_buenas = (indices == 0).sum()
reto_moderadas = (indices == 1).sum()
reto_daninas_sensibles = (indices == 2).sum()
reto_daninas_salud = (indices == 3).sum()

print("Conteos con np.digitize:",
      reto_buenas, reto_moderadas, reto_daninas_sensibles, reto_daninas_salud)
print("Coinciden con el Ejercicio 7:",
      (reto_buenas, reto_moderadas, reto_daninas_sensibles, reto_daninas_salud) ==
      (buenas, moderadas, daninas_sensibles, daninas_salud))

In [ ]:
# Celda de verificación — ejecútela al final, después de todo lo demás
assert lecturas_crudas.shape == (5, 7) and lecturas_crudas.ndim == 2, "revise lecturas_crudas del Ejercicio 1"
assert forma == (5, 7) and dimensiones == 2 and total == 35 and tipo == np.float64, "revise los atributos del Ejercicio 1"
assert np.array_equal(dias, np.arange(1, 8)), "dias debe ir del 1 al 7 (el 7 incluido)"
assert np.allclose(pendientes, [1.2, 0.9, 1.1, 1.0, 1.3]) and np.allclose(interceptos, [-0.5, 1.0, -2.0, 0.5, -1.5]), "revise pendientes e interceptos"
assert lectura_c_dia4 == 27.0 and ultimo_primer == 5.0, "revise la indexación del Ejercicio 2"
assert np.array_equal(dia5, [28.0, 41.0, 33.0, 52.0, 45.0]) and semana_b.shape == (7,), "revise dia5 y semana_b del Ejercicio 2"
assert bloque.shape == (3, 4) and bloque[0, 0] == 35.0, "revise el bloque del Ejercicio 2"
assert int(faltantes.sum()) == 2 and lecturas_utiles.shape == (5,), "revise las máscaras del Ejercicio 3"
assert abs(promedio_utiles - 27.0) < 1e-9, "revise promedio_utiles del Ejercicio 3"
assert calibradas.shape == (5, 7), "calibradas debe tener forma (5, 7)"
assert np.allclose(calibradas[0], [11.5, 16.9, 25.9, 36.1, 33.1, 21.1, 10.9]), "revise la calibración del Ejercicio 4"
assert lecturas_crudas[0, 0] == 10.0, "lecturas_crudas no debe modificarse"
assert abs(promedio_calibrado_sbc_a - 31.9) < 1e-9, "revise promedio_calibrado_sbc_a del Ejercicio 4"
assert abs(promedio_general - 24.1086) < 1e-4, "revise promedio_general del Ejercicio 5"
assert np.allclose(np.round(promedio_por_sensor, 2), [22.21, 26.07, 17.64, 31.79, 22.83]), "revise promedio_por_sensor (axis=1)"
assert np.allclose(np.round(promedio_por_dia, 2), [10.12, 16.14, 25.62, 35.92, 42.96, 27.94, 10.06]), "revise promedio_por_dia (axis=0)"
assert np.allclose(maximo_por_sensor, [36.1, 44.2, 34.3, 52.5, 57.0]), "revise maximo_por_sensor"
assert abs(media_sobre_limite - 43.275) < 1e-9, "revise media_sobre_limite"
assert np.allclose(excedente_por_sensor, [0.7, 11.3, 0.0, 27.3, 23.7]), "revise excedente_por_sensor del Ejercicio 6"
assert np.allclose(desvio_sensor_b, [23.6, 16.4, 2.9, 8.8, 2.5, 11.9, 21.8]), "revise desvio_sensor_b"
assert en_alerta.shape == (5, 7) and int((en_alerta > 0).sum()) == 8, "revise en_alerta"
assert abs(calibradas[0, 3] - 36.1) < 1e-9, "calibradas no debe modificarse al crear en_alerta"
assert (buenas, moderadas, daninas_sensibles, daninas_salud) == (9, 18, 7, 1), "revise los conteos del Ejercicio 7"
assert buenas + moderadas + daninas_sensibles + daninas_salud == calibradas.size, "los cuatro conteos deben sumar 35"
assert dia_pico == 5, "revise dia_pico del Ejercicio 7"
print("Verificación completada sin errores.")